# 事前準備：共通コードの実行
* このノートブックをコピーしたら，まずは以下の3つの共通コード（コードA～コードC）を順番に実行する
* これらの共通コードを実行しないと，それ以降のコードが実行できないので注意する
* また，コードA，コードB，及びコードC は，ノートブックに接続するたび毎回実行すること（ノートブックに接続中は，何度も実行する必要はない）
* 共通コードの詳細についての説明は割愛する（コードAとコードBの簡単な説明は[第2回のサンプルノートブック](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook02.ipynb)を参照）

In [ ]:
# コードA：日本語化ライブラリ導入
! pip install japanize-matplotlib | tail -n 1

In [2]:
# コードB：共通事前処理

# B1:余分なワーニングを非表示にする
import warnings
warnings.filterwarnings('ignore')

# 必要ライブラリのimport
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib # matplotlib日本語化対応
import seaborn as sns

# B2:データフレーム表示用関数
from IPython.display import display

# B3:表示オプション調整
np.set_printoptions(suppress = True, precision = 3) #numpyの浮動小数点の表示精度
pd.options.display.float_format = '{:.3f}'.format #pandasでの浮動小数点の表示精度
pd.set_option('display.max_columns', None) #データフレームですべての列データを表示

# B4:グラフのデフォルトフォント指定
plt.rcParams['font.size'] = 14

# 乱数の種
random_seed = 0

In [ ]:
# コードC：データのダウンロード
!curl -L -o wholesale.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/10/wholesale.csv
!curl -L -o Boston.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/10/Boston.csv

# 目的と到達目標

## 目的
* 機械学習モデルの予測性能を評価するための代表的な指標を理解する
* 1回だけの訓練・テスト分割に依存しない評価方法を理解する

## 到達目標
* 回帰モデルの代表的な評価指標である MAE，MSE，RMSE の意味を説明できる
* 分類モデルの混同行列，適合率，再現率，F1-score の意味を説明できる
* K分割交差検証の目的と手順を説明できる
* 基本的な予測性能評価を実践できる

# 予測性能評価

## なぜ予測性能を評価するのか
* 機械学習では，モデルを学習させるだけでは不十分である
* 学習済みモデルが，未知のデータに対してどの程度正しく予測できるかを確認する必要がある
* そのために，学習済みモデルによる予測結果と正解データを比較して評価指標を計算する
* 評価指標は，回帰と分類で異なる
  * 回帰：予測値と正解値の「ずれ」を評価
  * 分類：正しく分類できたか，どのように間違えたかを評価

## 今回扱う内容
* 回帰の予測性能評価
  * MAE（Mean Absolute Error: 平均絶対誤差）
  * MSE（Mean Squared Error: 平均2乗誤差）
  * RMSE（Root Mean Squared Error: 二乗平均平方根誤差）
* 分類の予測性能評価
  * 混同行列（Confusion Matrix）
  * 適合率（Precision）
  * 再現率（Recall / Sensitivity）
  * F1-score
* K分割交差検証（K-fold cross-validation）

# 回帰の予測性能評価

## 回帰における「誤差」
* 回帰では，正解値と予測値の差を使って予測性能を評価する
* 例：
  * 正解値が 10，予測値が 8 ⇒ 誤差の大きさは 2
  * 正解値が 10，予測値が 13 ⇒ 誤差の大きさは 3
* 正負の誤差をそのまま平均すると打ち消し合うため，絶対値や2乗を利用する

## MAE（平均絶対誤差）
* MAE（Mean Absolute Error）：予測誤差の絶対値を平均したもの
* 予測値と正解値が近いほど MAE は小さくなる
* MAE が 0 のとき，すべての予測値が正解値と一致している
* 元の目的変数と同じ単位で解釈できる
* MAEは，`sklearn.metrics` モジュールの `mean_absolute_error`関数で計算できる

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `mean_absolute_error` 関数をインポートする
* 3行目: 正解値をリスト `y_true` に代入
* 4行目: 予測値をリスト `y_pred` に代入
* 5行目: `mean_absolute_error` 関数で，`y_true` と `y_pred` の MAE を計算し，変数 `mae` に代入
* 6行目: `print` 関数とf-stringを使って，MAE を小数点以下2桁まで表示

In [10]:
# MAEの計算
from sklearn.metrics import mean_absolute_error
y_true = [10, 20, 30, 40]
y_pred = [12, 18, 33, 37]
mae = mean_absolute_error(y_true, y_pred)
print(f'MAE: {mae:.2f}')

MAE: 2.50


## MSE（平均2乗誤差）
* MSE（Mean Squared Error）：予測誤差を2乗して平均したもの
* 誤差を2乗するため，大きな誤差の影響を強く受ける
* MSE が小さいほど，予測値と正解値が近い
* 誤差を2乗しているため，値の単位も2乗される
* MSEは，`sklearn.metrics` モジュールの `mean_squared_error`関数で計算できる

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `mean_squared_error` 関数をインポートする
* 3行目: 正解値をリスト `y_true` に代入
* 4行目: 予測値をリスト `y_pred` に代入
* 5行目: `mean_squared_error` 関数で，`y_true` と `y_pred` の MSE を計算し，変数 `mse` に代入
* 6行目: `print` 関数とf-stringを使って，MSE を小数点以下2桁まで表示

In [9]:
# MSEの計算
from sklearn.metrics import mean_squared_error
y_true = [10, 20, 30, 40]
y_pred = [12, 18, 33, 37]
mse = mean_squared_error(y_true, y_pred)
print(f'MSE: {mse:.2f}')

MSE: 6.50


## RMSE（二乗平均平方根誤差）
* RMSE（Root Mean Squared Error）：MSE の平方根
* MSE と同様に，大きな誤差の影響を強く受ける
* 平方根をとるため，目的変数と同じ単位で解釈できる
* MAE と RMSE を比較すると，RMSE のほうが大きな誤差をより強く反映する

**［以下のコードの処理内容］**
* 2行目: 平方根を計算するために `math` モジュールの`sqrt`関数をインポートする
* 3行目: 正解値をリスト `y_true` に代入
* 4行目: 予測値をリスト `y_pred` に代入
* 5行目: `mean_squared_error` 関数で，`y_true` と `y_pred` の MSE を計算し，変数 `mse` に代入
* 6行目: `sqrt` 関数で MSE の平方根を計算し，変数 `rmse` に代入
* 7行目: `print` 関数とf-stringを使って，RMSE を小数点以下2桁まで表示

In [11]:
# RMSEの計算
from math import sqrt
y_true = [10, 20, 30, 40]
y_pred = [12, 18, 33, 37]
mse = mean_squared_error(y_true, y_pred)
rmse = sqrt(mse)
print(f'RMSE: {rmse:.2f}')

RMSE: 2.55


## MAE・MSE・RMSEの比較
| 指標 | 定義 | 元の目的変数と同じ単位 | 大きな誤差を強く評価 |
|:--|:--|:--|:--|
| MAE | 誤差の絶対値の平均 | ✓ | - |
| MSE | 誤差の2乗の平均 | - | ✓ |
| RMSE | MSEの平方根 | ✓ | ✓ |

* どの指標も，小さいほど予測性能がよい
* 指標によって誤差の評価のしかたが異なるため，目的に応じて使い分ける

### 実習：MAE・MSE・RMSEの計算
* 以下の正解値と予測値に対して MAE，MSE，RMSE を計算する
  * 正解値: `[10, 20, 30, 40]`
  * 予測値: `[12, 18, 33, 71]`
* この例では，実際の値が「40」のときに「71」と予測しており，大きく外れている
* このような場合，MAE・MSE・RMSEにどのような影響が出るのかを確認する

**［実習内容］**
* 「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2～4行目: MAE，MSE，平方根を計算するために必要な関数をインポートする
* 5行目: 正解値を `y_true` に代入
* 6行目: 予測値を `y_pred` に代入
* 7行目: `y_true` と `y_pred` の MAE を計算し，変数 `mae` に代入
* 8行目: `y_true` と `y_pred` の MSE を計算し，変数 `mse` に代入
* 96行目: RMSE（MSEの平方根）を計算し，変数 `rmse` に代入
* 10～12行目: `print` 関数とf-stringを使って，MAE，MSE，RMSE を小数点以下2桁までそれぞれ表示

**［実行結果］**
```
MAE: 9.50
MSE: 244.50
RMSE: 15.64
```

In [ ]:
# MAE・MSE・RMSEの計算
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from math import sqrt
y_true = [10, 20, 30, 40]




print(f'MAE: {mae:.2f}')
print(f'MSE: {mse:.2f}')
print(f'RMSE: {rmse:.2f}')

MAE: 9.50
MSE: 244.50
RMSE: 15.64


# 分類の予測性能評価

## 分類では「間違い方」も重要
* 分類では，正しく分類できた割合だけを見ればよいとは限らない
* 特に，2カテゴリ分類では，どちらか一方を**対象カテゴリ**として考える
  * 対象カテゴリを「陽性」，それ以外のカテゴリを「陰性」と呼ぶこともある
* 例えば，客船の生存予測では，「生存」を対象カテゴリとすることができる
* このとき，次の2種類の間違いを区別することが重要である
  * 本当は対象カテゴリなのに，それ以外のカテゴリと予測する
  * 本当はそれ以外のカテゴリなのに，対象カテゴリと予測する
* どちらの間違いを重視するかは，分析の目的によって異なる

## 混同行列
* 混同行列：正解カテゴリと予測カテゴリの組み合わせごとの件数を表した表
* 2カテゴリ分類では，次の4種類に分けて考える
  * **True Positive（TP）**：対象カテゴリを対象カテゴリと正しく予測
  * **True Negative（TN）**：それ以外のカテゴリをそれ以外のカテゴリと正しく予測
  * **False Positive（FP）**：それ以外のカテゴリを対象カテゴリと誤って予測
  * **False Negative（FN）**：対象カテゴリをそれ以外のカテゴリと誤って予測
* 混同行列は，`sklearn.metrics` モジュールの `confusion_matrix` 関数で計算できる
  * 書式: `confusion_matrix([正解カテゴリのリスト], [予測カテゴリのリスト])`

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `confusion_matrix` 関数をインポート
* 3行目: 正解カテゴリをリスト `t` に代入
  * `1` を対象カテゴリ，`0` をそれ以外のカテゴリとした（予測カテゴリも同様）
* 4行目: 予測カテゴリをリスト `pred` に代入
* 5行目: `confusion_matrix` 関数で混同行列を計算し，変数 `cm` に代入
* 6行目: `DataFrame` 関数で混同行列に行名・列名を付け，`display` 関数で表示（下表）

|  | 予測0 | 予測1 |
|---|---:|---:|
| 正解0 | **TN** | **FP** |
| 正解1 | **FN** | **TP** |

In [20]:
# 混同行列の計算
from sklearn.metrics import confusion_matrix
t = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
pred = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
cm = confusion_matrix(t, pred)
display(pd.DataFrame(cm, index=['正解0', '正解1'], columns=['予測0', '予測1']))

,予測0,予測1
正解0,5,1
正解1,2,2


## 適合率（Precision）
* 適合率：モデルが**対象カテゴリと予測したデータのうち，実際に対象カテゴリだった割合**
  * $\displaystyle{\mathrm{Precision}=\frac{TP}{TP+FP}}$
* 「対象カテゴリと予測した結果を，どの程度信用できるか」を表す
* 対象カテゴリではないデータを，対象カテゴリと誤って予測することを少なくしたい場合に重要となる
* つまり，**False Positive（FP）を少なくしたい場合**に重視される
* 適合率は，`sklearn.metrics` モジュールの `precision_score` 関数で計算できる
  * 書式: `precision_score([正解カテゴリのリスト], [予測カテゴリのリスト])`

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `precision_score` 関数をインポート
* 3行目: 正解カテゴリをリスト `t` に代入
* 4行目: 予測カテゴリをリスト `pred` に代入
* 5行目: `precision_score` 関数で適合率を計算し，変数 `precision` に代入
* 6行目: `print` 関数とf-stringを使って，適合率（`precision`）を小数点以下2桁まで表示

In [21]:
# 適合率の計算
from sklearn.metrics import precision_score
t = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
pred = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
precision = precision_score(t, pred)
print(f'適合率: {precision:.2f}')

適合率: 0.67


## 再現率（Recall）
* 再現率：**実際に対象カテゴリであるデータのうち，正しく対象カテゴリと予測できた割合**
  * $\displaystyle{\mathrm{Recall}=\frac{TP}{TP+FN}}$
* 「対象カテゴリを，どの程度見逃さずに見つけられたか」を表す
* 対象カテゴリであるデータを，それ以外のカテゴリと誤って予測することを少なくしたい場合に重要となる
* つまり，**False Negative（FN）を少なくしたい場合**に重視される
* 適合率は，`sklearn.metrics` モジュールの `recall_score` 関数で計算できる
  * 書式: `recall_score([正解カテゴリのリスト], [予測カテゴリのリスト])`

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `recall_score` 関数をインポート
* 3行目: 正解カテゴリをリスト `t` に代入
* 4行目: 予測カテゴリをリスト `pred` に代入
* 5行目: `recall_score` 関数で適合率を計算し，変数 `recall` に代入
* 6行目: `print` 関数とf-stringを使って，適合率（`recall`）を小数点以下2桁まで表示

In [22]:
# 再現率の計算
from sklearn.metrics import recall_score
t = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
pred = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
recall = recall_score(t, pred)
print(f'再現率: {recall:.2f}')

再現率: 0.50


## f1-score
* f1-score：適合率と再現率の両方を考慮した評価指標
  * $\displaystyle{\text{f1-score}=\frac{2\times \mathrm{Precision}\times \mathrm{Recall}}
{\mathrm{Precision}+\mathrm{Recall}}}$
* f1-scoreは，適合率と再現率の調和平均として計算される
* 調和平均とは，複数の値の平均を求める方法の一つで，一般的な平均値（算術平均）と比べて，**小さい値の影響を強く受ける** という特徴がある
* そのため，適合率と再現率のどちらか一方だけが高くても，F1-score は高くなりにくい
* 不均衡データの評価でもよく利用される
* f1-scoreは，`sklearn.metrics` モジュールの `f1_score` 関数で計算できる
  * 書式: `f1_score([正解カテゴリのリスト], [予測カテゴリのリスト])`

**［以下のコードの処理内容］**
* 2行目: `sklearn.metrics` モジュールから `f1_score` 関数をインポート
* 3行目: 正解カテゴリをリスト `t` に代入
* 4行目: 予測カテゴリをリスト `pred` に代入
* 5行目: `f1_score` 関数で f1-score を計算し，変数 `f1` に代入
* 6行目: `print` 関数とf-stringを使って，適合率（`f1`）を小数点以下2桁まで表示

In [25]:
# f1-scoreの計算
from sklearn.metrics import f1_score
t = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
pred = [1, 1, 0, 0, 1, 0, 0, 0, 0, 0]
f1 = f1_score(t, pred)
print(f'f1-score: {f1:.2f}')

f1-score: 0.57


# ■ここまで


## classification_report
* `classification_report` 関数を使うと，適合率，再現率，F1-score などをまとめて確認できる
* 複数カテゴリ分類でも，各カテゴリについて評価指標を確認できる

In [ ]:
# 評価指標をまとめて表示
from sklearn.metrics import classification_report

report = classification_report(t, pred)
print(report)

### コードの説明：classification_report
* 2行目: `classification_report` 関数を読み込む
* 4行目: `classification_report` 関数に正解データ `t` と予測結果 `pred` を渡し，結果を `report` に代入
* 5行目: `print` 関数で結果を表示
* 出力には，カテゴリごとの precision，recall，f1-score，support などが含まれる

### 実習2：分類の評価指標を計算
**［実習内容］**
* 以下の `t2` と `pred2` に対して，適合率，再現率，F1-score を計算する
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: 正解カテゴリを `t2` に代入
* 3行目: 予測カテゴリを `pred2` に代入
* 5～7行目: 適合率，再現率，F1-score をそれぞれ計算
* 9～11行目: 計算結果を表示

In [ ]:
# 実習2：分類の評価
t2 = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
pred2 = [1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0]





print('適合率:', precision2)
print('再現率:', recall2)
print('F1-score:', f1_2)

## 不均衡データと評価指標
* カテゴリごとのデータ数が大きく異なるデータを「不均衡データ」と呼ぶ
* 不均衡データでは，正解率だけを見るとモデルの性能を誤解することがある
* 例えば，多数派カテゴリだけを予測しても，全体の正解率が高くなる場合がある
* そのため，適合率，再現率，F1-score，混同行列なども確認することが重要である

# K分割交差検証

## ホールドアウト法の問題点
* これまでの講義では，データを訓練データとテストデータに1回だけ分割して評価してきた
* この方法をホールドアウト法と呼ぶ
* しかし，データの分け方によって評価結果が変わることがある
* 特にデータ数が少ない場合，1回だけの分割結果に評価が左右されやすい

## K分割交差検証の考え方
* K分割交差検証では，データを K 個のグループに分割する
* そのうち1つを検証用，残りを学習用として評価する
* 検証用に使うグループを入れ替えながら，K回の学習と評価を行う
* 最後に K 回の評価結果を平均して，モデルの性能を確認する
* 1回だけの分割より，安定した評価を行いやすい

## 実習用データ
* K分割交差検証の実習では，scikit-learnに付属する糖尿病データを利用する
* 正解データは数値なので，回帰問題として扱う
* データの読み込みに複雑な前処理が不要なので，交差検証の処理に集中できる

In [ ]:
# 糖尿病データの読み込み
from sklearn.datasets import load_diabetes

data = load_diabetes()
x = pd.DataFrame(data.data, columns=data.feature_names)
t = pd.Series(data.target)

display(x.head())
display(t.head())

### コードの説明：糖尿病データの読み込み
**［以下のコードの処理内容］**
* 2行目: `sklearn.datasets` モジュールから `load_diabetes` 関数を読み込む
* 4行目: `load_diabetes` 関数でデータを読み込み，変数 `data` に代入
* 5行目: 説明変数を DataFrame に変換し，変数 `x` に代入
* 6行目: 正解データを Series に変換し，変数 `t` に代入
* 8～9行目: 先頭5件を表示

## KFoldクラス
* `KFold` クラスを使うと，K分割交差検証におけるデータの分割方法を設定できる
* `n_splits=3`：データを3分割
* `shuffle=True`：分割前にデータをランダムに並べ替える
* `random_state=0`：ランダムな処理を再現可能にする

In [ ]:
# K分割交差検証の分割方法を設定
from sklearn.model_selection import KFold

kf = KFold(n_splits=3,
           shuffle=True,
           random_state=random_seed)

### コードの説明：KFold
* 2行目: `sklearn.model_selection` モジュールから `KFold` クラスを読み込む
* 4～6行目: `KFold` クラスのオブジェクトを生成し，変数 `kf` に代入
  * `n_splits=3` で3分割
  * `shuffle=True` でデータをランダムに並べ替える
  * `random_state=random_seed` で乱数を固定

## cross_validate関数
* `cross_validate` 関数を使うと，K分割交差検証による学習と評価をまとめて実行できる
* 主な引数
  * 第1引数：モデル
  * 第2引数：説明変数
  * 第3引数：正解データ
  * `cv`：分割方法
  * `scoring`：評価指標
* 戻り値には，各回の評価結果などが含まれる

In [ ]:
# 線形回帰モデルを3分割交差検証で評価
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate

model = LinearRegression()

result = cross_validate(model,
                        x,
                        t,
                        cv=kf,
                        scoring='r2')

display(pd.DataFrame(result))

### コードの説明：cross_validate
**［以下のコードの処理内容］**
* 2行目: `LinearRegression` クラスを読み込む
* 3行目: `cross_validate` 関数を読み込む
* 5行目: 線形回帰モデルのオブジェクトを生成し，変数 `model` に代入
* 7～11行目: `cross_validate` 関数で3分割交差検証を実行
  * 第1引数にモデル `model`
  * 第2引数に説明変数 `x`
  * 第3引数に正解データ `t`
  * `cv=kf` で分割方法を指定
  * `scoring='r2'` で決定係数を評価指標に指定
* 13行目: 結果を DataFrame に変換して表示

## 交差検証の平均値
* K分割交差検証では，各分割で得られた評価結果を確認する
* 最終的には，それらの平均値をモデルの評価の目安として利用する
* `cross_validate` の戻り値 `test_score` に各回の評価値が格納されている

In [ ]:
# 3回の評価結果と平均値を確認
print(result['test_score'])
print(result['test_score'].mean())

### コードの説明：評価値の平均
* 2行目: `result['test_score']` で各回の決定係数を表示
* 3行目: `mean` メソッドで3回の決定係数の平均値を計算して表示

### 実習3：5分割交差検証
**［実習内容］**
* これまでの3分割交差検証を5分割交差検証に変更する
* 評価指標は決定係数 `r2` とする
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: `KFold` クラスで5分割するためのオブジェクトを作成
* 3行目: `cross_validate` 関数で5分割交差検証を行う
* 4行目: 5回の `test_score` の平均値を表示

In [ ]:
# 実習3：5分割交差検証



print(result5['test_score'].mean())

## 分類における交差検証
* 分類では，各カテゴリのデータ数の偏りにも注意する
* 分割後の各グループでカテゴリの割合が大きく変わると，評価結果が不安定になることがある
* そのため，分類ではカテゴリの割合をなるべく保ちながら分割する方法が利用される
* scikit-learnでは `StratifiedKFold` を利用できる

In [ ]:
# StratifiedKFoldの例
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=3,
                      shuffle=True,
                      random_state=random_seed)

### コードの説明：StratifiedKFold
* 2行目: `StratifiedKFold` クラスを読み込む
* 4～6行目: 3分割するためのオブジェクトを生成
* 各カテゴリの割合をできるだけ保ちながら分割する

## 交差検証とチューニング
* モデルの設定値（ハイパーパラメータ）を変更しながら性能を比較することをチューニングと呼ぶ
* 同じテストデータを何度も見ながらチューニングすると，そのテストデータに合わせすぎる可能性がある
* そのため，交差検証を利用してチューニングを行い，最後に独立したテストデータで最終評価する考え方が重要である

# まとめ
* 回帰では，予測値と正解値のずれを MAE，MSE，RMSE などで評価する
* MAE は誤差の絶対値の平均，MSE は誤差の2乗の平均，RMSE は MSE の平方根
* 分類では，正解率だけでなく，混同行列，適合率，再現率，F1-score などを確認する
* 適合率と再現率は，どの種類の誤りを重視するかによって使い分ける
* K分割交差検証では，データの分け方を変えて複数回評価することで，1回だけの分割に依存しにくい評価ができる
* 分類では，カテゴリの割合を保つ `StratifiedKFold` が利用できる

# 到達目標の確認
* MAE，MSE，RMSE の違いを説明できますか？
* 適合率と再現率の違いを説明できますか？
* F1-scoreを利用する意味を説明できますか？
* K分割交差検証を行う理由を説明できますか？
* `cross_validate` 関数を使ってモデルの評価を実行できますか？